# CRISP-DM Fase 4: Data Modeling - Arquitectura Relacional y Lakehouse
## Dataset: SIPSA Precios de Insumos y Fertilizantes Agrícolas (DANE)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 4: Dimensional & Relational Modeling) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: Ralph Kimball Dimensional Architecture, Medallion Architecture (Gold Layer), Transacciones ACID.

---

### 1. Arquitectura Relacional y Persistencia Lakehouse (Gold Layer)
La **Capa Gold** del Lakehouse materializa los datos en una base de datos relacional estructurada ([`agrostats_lakehouse.db`](file:///c:/Users/ADAN/OneDrive/Documentos/AgroStatsApp/data/processed/agrostats_lakehouse.db)).
* **Modelo Dimensional de Kimball**: Separación entre **Tablas de Hechos** (*Fact Tables*, que registran eventos transaccionales como cotizaciones de precios o volúmenes de carga) y **Tablas de Dimensiones** (*Dimension Tables*, que contienen entidades maestras como Municipios DIVIPOLA o Estaciones Meteorológicas).
* **Consistencia Transaccional ACID**: Integridad atómica garantizada en cada operación de carga.
* **Integridad Cruzada**: Verificación estricta de paridad entre los conteos de registros de la capa columnar Parquet y el motor relacional SQL.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN ROBUSTA DE RUTAS DEL PROYECTO (PEP 8)]
# Compatible con Local Jupyter, VSCode, Colab, Databricks Git Folders & Repos
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Búsqueda dinámica ascendente de la raíz del proyecto (donde reside src/ y metadata.json)
CURRENT_DIR = Path(".").resolve()
APP_ROOT = None

for candidate in [CURRENT_DIR, CURRENT_DIR.parent, CURRENT_DIR.parent.parent, CURRENT_DIR.parent.parent.parent]:
    if (candidate / "src" / "database").exists() or (candidate / "metadata.json").exists():
        APP_ROOT = candidate
        break

if APP_ROOT is None:
    candidate = CURRENT_DIR
    while candidate.parent != candidate:
        if (candidate / "src" / "database").exists():
            APP_ROOT = candidate
            break
        candidate = candidate.parent

if APP_ROOT is None:
    APP_ROOT = CURRENT_DIR

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno e importaciones inicializadas exitosamente.")
print(f"📍 Raíz del proyecto detectada: {APP_ROOT}")


### 2. Carga Idempotente (Upsert) en el Lakehouse
Utilizamos `DatabaseManager` para gestionar la conexión a SQLite, crear la tabla con el esquema derivado y reemplazar de forma controlada el contenido sin bloqueos.


In [ ]:
# ==============================================================================
# [FASE 6: CARGA IDEMPOTENTE AL DATA LAKEHOUSE (SQLITE GOLD)]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
db = DatabaseManager(str(db_path))

parquet_file = APP_ROOT / "data/processed/sipsa_insumos.parquet"
df = pd.read_parquet(parquet_file)

table_name = "sipsa_insumos"
print(f"🗄️ Persistiendo {len(df):,} registros en tabla SQL '{table_name}'...")
db.save_dataframe(df, table_name, if_exists="replace")

# Inspección de esquema SQL resultante
schema_info = db.query(f"PRAGMA table_info('{table_name}')")
print("\n📋 Esquema Relacional de la Tabla:")
display(schema_info)


### 3. Consultas Analíticas SQL y Control de Paridad
Ejecutamos consultas SQL agregadas para verificar la consistencia de los datos en base de datos y comprobamos que no exista pérdida de registros respecto a la capa Parquet:
$$\text{COUNT}_{\text{SQL}} = N_{\text{Parquet}}$$


In [ ]:
# ==============================================================================
# [FASE 6: CONSULTAS ANALÍTICAS Y CONTROL DE INTEGRIDAD PARQUET vs SQL]
# ==============================================================================
primary_col = "indice_total"
group_col = "grupo_producto"

# Query analítica de agregación
if primary_col in df.columns and pd.api.types.is_numeric_dtype(df[primary_col]):
    query = "SELECT COUNT(*) AS total_registros, ROUND(AVG(indice_total), 2) AS promedio, ROUND(MIN(indice_total), 2) AS minimo, ROUND(MAX(indice_total), 2) AS maximo FROM sipsa_insumos"
    agg_res = db.query(query)
    print("📈 Resumen Agregado SQL:")
    display(agg_res)

# Verificación de paridad de registros (Parquet == SQL)
count_sql = db.query("SELECT COUNT(*) AS c FROM sipsa_insumos").iloc[0]['c']
count_parquet = len(df)
assert count_sql == count_parquet, f"Error: Desalineación entre Parquet ({count_parquet}) y SQL ({count_sql})"
print(f"✅ Integridad verificada al 100%: {count_sql:,} registros consistentes en Lakehouse.")


### 4. Conclusión de Modelado Dimensional
* **Resultado**: Tabla persistida, indexada y consultable mediante SQL estándar con paridad total de registros.
* **Siguiente Paso**: Continuar a `07_modeling_and_integration.ipynb` para ejecutar la inferencia estadística dual y resolver las preguntas de negocio A1-J1.
